# Pipeline results and stage outputs: E0–E7

Read-only analysis of scaling, imagination, assembly, gating, self-training, and robustness runs. Set the paths below, then **Run All**. Re-run the loading cell or use **Refresh files** to see new results while experiments run.

The notebook reads saved outputs; it never launches models, trains, changes study identities, or writes evaluation results. Oracle and smoke outputs are excluded from research charts by default. Missing evaluations remain unscored. Stage execution status is **not** a scientific pass/fail decision.

In [ ]:
from pathlib import Path
import os, sys, json

# Individual runs OR parent directories; nested parallel shard roots are supported.
RUN_PATHS = [Path(os.environ.get("GA_RESULTS_ROOT", str(Path(os.environ.get("GA_ROOT", "/home/kpandey/generative_assembly")) / "runs")))]
# Example: RUN_PATHS = ["/home/kpandey/generative_assembly/runs/imagination-v2-...", "/home/kpandey/generative_assembly/runs/stage4-full-n4096"]
# Downloaded runs may need explicit local dataset bindings (for reference previews only).
DATASETS = {}  # {"/local/run": "/local/data/dataset.json"}
SPLIT = "dev"  # train, dev, test, real; None displays all saved splits
INCLUDE_ORACLES = False
INCLUDE_SMOKE = False
INTERACTIVE_3D = True
MAX_POINTS_PER_CLOUD = 2500
MAX_IMAGES = 24
ROWS_PER_PLOT = 30  # paginate large multi-shard comparisons

# Works from either notebook location, including on the Linux experiment host.
candidates = [Path.cwd(), Path.cwd() / "pt_reg", Path.cwd().parent / "pt_reg"]
helper_root = next((p for p in candidates if (p / "pipeline_results_viewer.py").is_file()), None)
if helper_root is None:
    raise FileNotFoundError("Keep pipeline_results_viewer.py beside this notebook or in ./pt_reg; run from the repository root or pt_reg.")
sys.path.insert(0, str(helper_root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from pipeline_results_viewer import (Catalog, STAGES, POSE_STAGES, outcome_table, comparison_table,
    image_gallery, point_panels, point_figure, learning_curves, browser, read_json)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 80)
try:
    import plotly
except ImportError:
    INTERACTIVE_3D = False
    print("Plotly unavailable: using Matplotlib 3D previews.")
print("Run paths:", *map(str, RUN_PATHS), sep="\n")

## Load and refresh

Reading an active run is supported. Atomic job records are preferred over generated indexes; malformed/incomplete files are listed below. Locks mean **liveness unknown**, including locks left after a worker crash. No GPU access or checkpoint loading is required.

In [ ]:
catalog = Catalog(RUN_PATHS, DATASETS)
all_jobs = catalog.frame
jobs = catalog.select(split=SPLIT, include_oracles=INCLUDE_ORACLES, include_smoke=INCLUDE_SMOKE)
print(f"{len(catalog.runs)} run(s), {len(all_jobs)} observed job(s), {len(jobs)} in the analysis selection.")
if INCLUDE_SMOKE:
    print("SMOKE OUTPUTS INCLUDED: software validation, not research evidence.")
if INCLUDE_ORACLES:
    print("ORACLE OUTPUTS INCLUDED: privileged diagnostics, not deployable results.")
for warning in catalog.warnings[:30]:
    print("READ WARNING:", warning)
if not len(jobs):
    print("No jobs in this selection. Configure RUN_PATHS, check SPLIT, and inspect smoke/oracle flags below.")
columns = ["run", "stage", "split", "status", "evaluation_status", "oracle", "smoke"]
display(all_jobs.groupby(columns, dropna=False).size().rename("observed_jobs").reset_index())

In [ ]:
def show_table(frame, columns=None):
    if frame.empty:
        print("No saved data available for this view.")
    else:
        display(frame[[c for c in columns if c in frame]] if columns else frame)

def scalar_summary(frame, metric):
    if metric not in frame or frame.empty:
        return pd.DataFrame()
    part = frame[frame.status.eq("complete")].copy()
    part[metric] = pd.to_numeric(part[metric], errors="coerce")
    part = part.dropna(subset=[metric, "source_id"])
    # Repeats, seeds, and fracture patterns do not become independent objects.
    per_source = part.groupby(["run", "stage", "arm", "source_id"], dropna=False)[metric].mean()
    return per_source.groupby(level=["run", "stage", "arm"]).agg(["mean", "count"]).reset_index()

def metric_chart(frame, metric, title):
    values = scalar_summary(frame, metric)
    if values.empty:
        print(f"{title}: no saved scored values.")
        return
    for start in range(0, len(values), ROWS_PER_PLOT):
        page = values.iloc[start:start+ROWS_PER_PLOT]
        fig, ax = plt.subplots(figsize=(12, max(3, len(page)*.3)))
        labels = [f"{Path(r.run).name} | {r.arm}" for _, r in page.iterrows()]
        ax.barh(range(len(page)), page["mean"], color="#2563eb")
        ax.set_yticks(range(len(page)), labels)
        ax.invert_yaxis()
        ax.set_xlim(0, max(float(values["mean"].max())*1.05, 1e-6))
        ax.set_xlabel(f"Source-macro mean: {metric}")
        ax.set_title(f"{title} | rows {start+1}-{start+len(page)} of {len(values)}")
        fig.tight_layout()
        plt.show()

def success_chart(frame, title):
    values = outcome_table(frame)
    if values.empty or values.source_macro_success.notna().sum() == 0:
        print(f"{title}: no saved pose outcomes; browse artifacts below.")
        return
    values = values.dropna(subset=["source_macro_success"])
    for start in range(0, len(values), ROWS_PER_PLOT):
        page = values.iloc[start:start+ROWS_PER_PLOT]
        fig, ax = plt.subplots(figsize=(12, max(3, len(page)*.3)))
        labels = [f"{Path(r.run).name} | {r.arm}" for _, r in page.iterrows()]
        ax.barh(range(len(page)), page.source_macro_success, color="#16a34a")
        ax.set_yticks(range(len(page)), labels)
        ax.set_xlim(0, 1)
        ax.invert_yaxis()
        ax.set_xlabel("Source-macro success (observed scored/failed pose jobs)")
        ax.set_title(f"{title} | rows {start+1}-{start+len(page)} of {len(values)}")
        fig.tight_layout()
        plt.show()
    show_table(values)

stage_names = {"E0": "Coordinate/render integrity", "E1": "Image completion", "E2": "3D reconstruction",
    "E3": "Template utility/scaling", "E4": "Generative assembly", "E5": "Uncertainty gating",
    "E6_TRAIN": "Student training", "E6": "Student inference", "E7": "Robustness"}
status = jobs.groupby(["stage", "status"]).size().unstack(fill_value=0)
if not status.empty:
    status.reindex([s for s in STAGES if s in status.index]).plot.barh(stacked=True, figsize=(11, 5), title="Observed job status by stage")
    plt.xlabel("Jobs (not independent sources)")
    plt.tight_layout()
    plt.show()
show_table(outcome_table(jobs))

## E0: inputs, rendering, coordinates, and candidates

Inspect the actual RGB render, depth/control image, and edit masks. The case browser below also shows initial candidate assemblies and coordinate-check JSON. Fragments have distinct colors; a template is always auxiliary geometry.

In [ ]:
e0 = jobs[jobs.stage.eq("E0")]
show_table(e0, ["run", "case_id", "arm", "status", "N", "seconds", "artifact_dir"])
INPUT_CASE = e0.case_id.iloc[0] if len(e0) else None
if INPUT_CASE:
    print("Example input case:", INPUT_CASE)
    display(HTML(image_gallery(e0[e0.case_id.eq(INPUT_CASE)], MAX_IMAGES)))

## E1: every completion and preservation evidence

Compare raw versus cleaned output, models, F/A inputs, views, and seeds. Silhouette IoU and drift are evaluator metrics; foreground growth and selection scores are input-only heuristics. A visually attractive completion does not establish assembly accuracy.

In [ ]:
e1 = jobs[jobs.stage.isin(["E1", "E1_ORACLE"])]
show_table(e1, ["run", "case_id", "arm", "model", "input_type", "seed", "status", "metric.silhouette_iou", "metric.exterior_contour_drift", "selection.growth_relative_to_input", "seconds"])
metric_chart(e1, "metric.silhouette_iou", "E1 intact silhouette IoU")
metric_chart(e1, "metric.exterior_contour_drift", "E1 exterior contour drift (lower is better)")
IMAGE_CASE = e1.case_id.iloc[0] if len(e1) else None
if IMAGE_CASE:
    display(HTML(image_gallery(e1[e1.case_id.eq(IMAGE_CASE)], MAX_IMAGES)))

## E2: bridge outputs, alignment, and rejected priors

The browser renders raw and aligned `shape.npz` surface samples, labels their frames, shows the observed anchor, and exposes `mesh.obj`, alignment metadata, and reconstruction logs in the artifact directory. Rejected/unaligned surfaces are never portrayed as valid aligned geometry.

In [ ]:
e2 = jobs[jobs.stage.eq("E2")]
show_table(e2, ["run", "case_id", "arm", "status", "template_rejected", "alignment.rejected", "alignment.fit_error", "alignment.heldout_error", "metric.shape_chamfer", "metric.fscore_001", "seconds", "artifact_dir"])
metric_chart(e2, "metric.shape_chamfer", "E2 full-shape error (lower is better)")
metric_chart(e2, "alignment.heldout_error", "E2 held-out observed compatibility (lower is better)")

## E3: scaling and local-basin diagnostics

E3 is privileged diagnostics, so set `INCLUDE_ORACLES=True` and reload to inspect these results. Compare candidate coverage and initial-to-final drift; do not interpret exact/perturbed GT starts as unknown-pose success. Different run settings remain separate.

In [ ]:
e3 = jobs[jobs.stage.eq("E3")]
show_table(e3, ["run", "N", "case_id", "arm", "oracle", "metric.success", "metric.candidate_recall", "metric.best_candidate_error", "metric.initially_successful", "metric.max_part_chamfer", "seconds"])
success_chart(e3, "E3 oracle template utility")
if len(e3) and "N" in e3 and "metric.success" in e3:
    per_source = e3[e3.status.eq("complete")].groupby(["N", "arm", "source_id"])["metric.success"].mean().reset_index()
    scaling = per_source.groupby(["N", "arm"])["metric.success"].mean().unstack()
    if not scaling.empty:
        scaling.plot(marker="o", figsize=(11, 5), title="E3 source-macro success vs candidate budget")
        plt.xscale("log", base=2)
        plt.ylabel("Success")
        plt.tight_layout()
        plt.show()

## E4: end-to-end arms and refinement damage

Compare B0–B6, completion models, F/A inputs, and rerank/refine arms. Per-prior arms are shown separately; this notebook does not choose the best prior using reference scores. B3 in historical runs is the same-category control. B5 is considered compute matched only when its saved flag says so.

In [ ]:
e4 = jobs[jobs.stage.eq("E4")]
success_chart(e4, "E4 observed outcomes by arm")
metric_chart(e4, "metric.max_part_chamfer", "E4 worst-fragment error")
show_table(e4, ["run", "case_id", "arm", "status", "metric.success", "metric.rotation_degrees", "metric.translation_error", "diagnostic.template_accepted", "diagnostic.refined", "diagnostic.contact_before", "diagnostic.contact_after", "diagnostic.compute_matched", "diagnostic.target_seconds", "diagnostic.search_candidates"])
comparisons = comparison_table(catalog, SPLIT)
if not comparisons.empty:
    comparisons = comparisons[comparisons.run.isin(jobs.run.unique())]
if not comparisons.empty:
    show_table(comparisons[comparisons.get("stage", pd.Series(None, index=comparisons.index, dtype="object")).eq("E4")])

## E5: K budgets, gate coverage, and safety

Acceptance coverage uses saved gate decisions. K counts, available hypotheses, and incomplete budgets are shown explicitly. Agreement and compatibility are heuristic trust evidence, not calibrated confidence. Paired damage and CI values below come from the saved evaluator summaries.

In [ ]:
e5 = jobs[jobs.stage.eq("E5")]
success_chart(e5, "E5 selected output outcomes")
show_table(e5, ["run", "case_id", "arm", "status", "policy", "diagnostic.requested_hypotheses", "diagnostic.available_hypotheses", "diagnostic.budget_complete", "diagnostic.template_accepted", "diagnostic.exterior_error", "diagnostic.contact_after", "metric.success"])
if "diagnostic.template_accepted" in e5:
    coverage = e5[e5.status.eq("complete")].groupby(["run", "arm", "source_id"])["diagnostic.template_accepted"].mean()
    coverage_table = coverage.groupby(level=["run", "arm"]).mean().rename("acceptance_coverage_source_macro").reset_index()
    show_table(coverage_table)
    if not coverage_table.empty:
        fig, ax = plt.subplots(figsize=(11, max(3, len(coverage_table)*.25)))
        ax.barh(range(len(coverage_table)), coverage_table.acceptance_coverage_source_macro, color="#f59e0b")
        ax.set_yticks(range(len(coverage_table)), [f"{Path(r.run).name} | {r.arm}" for _, r in coverage_table.iterrows()])
        ax.set(xlim=(0,1), xlabel="Source-macro acceptance coverage", title="E5 gate coverage (heuristic)")
        fig.tight_layout()
        plt.show()
if not comparisons.empty:
    show_table(comparisons[comparisons.get("stage", pd.Series(None, index=comparisons.index, dtype="object")).eq("E5")])

## E6: pseudo-labels, training, and students

Display accepted/rejected label records, source exposure, learning curves, and student-versus-control/teacher comparisons. Training loss is not an accuracy metric. Checkpoints are listed but never unpickled. Real training results may be absent until E4/E5 justify this stage.

In [ ]:
labels = pd.DataFrame(catalog.labels)
if not labels.empty:
    labels = labels[labels.run.isin(jobs.run.unique())]
show_table(labels, ["run", "case_id", "source_id", "accepted", "teacher_job", "gate_job", "oracle", "smoke"])
if not labels.empty:
    show_table(labels.groupby(["run", "accepted"]).agg(cases=("case_id", "size"), sources=("source_id", "nunique")).reset_index())
quality_rows = []
for run in jobs.run.unique():
    path = Path(run) / "evaluation" / "train" / "pseudo_label_quality.json"
    if path.exists():
        for item in read_json(path).get("labels", []):
            quality_rows.append({"run": run, "case_id": item.get("case_id"), "accepted": item.get("accepted"), **(item.get("metrics") or {})})
show_table(pd.DataFrame(quality_rows), ["run", "case_id", "accepted", "success", "max_part_chamfer"])
curves = learning_curves(catalog)
if not curves.empty:
    curves = curves[curves.run.isin(jobs.run.unique())]
if not curves.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    for (run, arm), part in curves.groupby(["run", "arm"]):
        part = part.sort_values("step").drop_duplicates("step", keep="last")
        ax.plot(part.step, part.loss, label=f"{Path(run).name} | {arm}")
    ax.set(xlabel="Optimizer update", ylabel="Loss", title="E6 learning curves (not assembly accuracy)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
e6 = jobs[jobs.stage.eq("E6")]
success_chart(e6, "E6 student outcomes")
if not comparisons.empty:
    show_table(comparisons[comparisons.get("stage", pd.Series(None, index=comparisons.index, dtype="object")).eq("E6")])

## E7: robustness and domain shift

Compare arms **within** each corruption factor, value, and run. Repeats are averaged within original sources. Missing-piece identities remain visible. The current erosion condition is a point-removal proxy. Reference-free real scans are unscored. Missing 3/5-piece patterns or real scans are not synthesized by the notebook.

In [ ]:
e7 = jobs[jobs.stage.eq("E7")]
show_table(e7, ["run", "base_case_id", "case_id", "arm", "factor.factor", "factor.value", "factor.repeat", "status", "evaluation_status", "metric.success", "diagnostic.template_accepted", "metric.max_part_chamfer"])
if len(e7):
    factor_groups = ["run", "stage", "arm"] + [c for c in ["factor.factor", "factor.value"] if c in e7]
    robust_outcomes = outcome_table(e7, factor_groups)
    show_table(robust_outcomes)
    if not robust_outcomes.empty and {"factor.factor", "factor.value"}.issubset(robust_outcomes.columns):
        for (run, factor), part in robust_outcomes.groupby(["run", "factor.factor"], dropna=False):
            values = part.dropna(subset=["source_macro_success"]).pivot(index="factor.value", columns="arm", values="source_macro_success")
            if not values.empty:
                values.plot.bar(figsize=(9, 3), title=f"E7 {Path(run).name} | {factor}")
                plt.ylim(0,1)
                plt.ylabel("Source-macro success")
                plt.xlabel("Corruption level (fixed factor)")
                plt.tight_layout()
                plt.show()
if not comparisons.empty:
    show_table(comparisons[comparisons.comparison_type.eq("robustness_comparisons")])
real_jobs = catalog.select(split="real", include_oracles=INCLUDE_ORACLES, include_smoke=INCLUDE_SMOKE)
show_table(real_jobs, ["run", "stage", "case_id", "arm", "status", "evaluation_status", "diagnostic.contact_after", "artifact_dir"])

## Runtime, saved gates, and failures

Durations are per-job measurements; summing them is work performed, not elapsed parallel-suite time. Shared cached jobs can occur in multiple roots. Saved gates are displayed with their run/split context; the notebook does not certify an incomplete study.

In [ ]:
if len(jobs):
    runtime = jobs.groupby(["run", "stage"]).seconds.agg(["count", "sum", "median", "max"]).reset_index()
    show_table(runtime)
gates = []
for entry in catalog.summaries:
    if (SPLIT is None or entry["split"] == SPLIT) and entry["run"] in jobs.run.unique():
        summary = entry["summary"]
        gates.append({"run": entry["run"], "split": entry["split"], "E4_gate": summary.get("E4_primary_gate"), "E1_gate": summary.get("E1_gate"), "E2_gate": summary.get("E2_gate"), "E6_gate": summary.get("E6_gate"), "limitations": summary.get("limitations")})
show_table(pd.DataFrame(gates))
failures = jobs[jobs.status.isin(["failed", "locked"])]
show_table(failures, ["run", "stage", "case_id", "arm", "status", "error", "artifact_dir"])
if "not_applicable" in jobs:
    show_table(jobs[jobs.not_applicable.notna()], ["run", "stage", "case_id", "arm", "not_applicable", "artifact_dir"])

## Suite-wide results across all shards

These are the scheduler's aggregated per-condition reports, with cached outcomes deduplicated and all eligible sources combined. Use these paired intervals for architecture decisions; individual worker reports cover only their shard. Oracle controls stay explicitly labelled. Run `python -m remaining_studies aggregate --root ...` outside this notebook if aggregate files have not yet been produced.

In [ ]:
suite_roots = set()
for value in RUN_PATHS:
    path = Path(value).expanduser().resolve()
    for candidate in [path] + list(path.parents)[:3]:
        if (candidate / "plan.json").is_file() and (candidate / "analysis").is_dir():
            suite_roots.add(candidate)
suite_groups, suite_comparisons, suite_gates = [], [], []
for suite in sorted(suite_roots):
    for path in sorted((suite / "analysis").glob("*/summary.json")):
        name = path.parent.name
        split = name.rsplit("__", 1)[-1]
        if SPLIT is not None and split != SPLIT:
            continue
        value = read_json(path)
        if value.get("smoke", False) and not INCLUDE_SMOKE:
            continue
        for item in value.get("groups", []):
            if item.get("oracle", False) and not INCLUDE_ORACLES:
                continue
            suite_groups.append({"suite": str(suite), "condition_split": name, **item})
        for category in ["paired_comparisons", "targeted_comparisons", "robustness_comparisons"]:
            for item in value.get(category, []):
                suite_comparisons.append({"condition_split": name, "comparison_type": category, **item})
        suite_gates.append({"condition_split": name, "E4_E5": value.get("gates", {}).get("reason"), "E6": value.get("E6_gate", value.get("E6_geometry_gain_gate"))})
show_table(pd.DataFrame(suite_groups))
show_table(pd.DataFrame(suite_comparisons))
show_table(pd.DataFrame(suite_gates))

## Interactive stage-output browser

Choose **run → stage → case → output**. Browse every completion seed and every pose arm, check dependencies, inspect diagnostics/logs, and rotate 3D surfaces. **Refresh files** reloads partial runs. **Check selected hashes** verifies the selected job only.

The reference toggle is evaluator-only. Prior geometry is gray; measured pieces keep their fragment colors. Initial E0 proposal and final poses share normalized coordinates. E7 rotation outputs are shown with base observations because the saved predictions were already mapped back.

If widgets are unavailable, the manual cell below provides the same image and 3D views.

In [ ]:
try:
    import ipywidgets
    display(browser(catalog, include_smoke=INCLUDE_SMOKE, interactive=INTERACTIVE_3D, max_points=MAX_POINTS_PER_CLOUD))
except ImportError:
    print("ipywidgets unavailable; use the manual case preview below.")

## Manual case preview / static export

Set `SELECTED_RUN`, `SELECTED_JOB`, and `SHOW_REFERENCE`, then rerun this cell. By default it selects a completed E4/E5/E6/E7 prediction if available, then E2 or E0. Images and interactive 3D HTML embed their data so saved output can be viewed away from the inference host. The full-resolution mesh and point files remain in the job directory.

In [ ]:
preview_jobs = jobs[jobs.status.eq("complete") & jobs.stage.isin(["E4", "E5", "E6", "E7", "E2", "E0"])]
preferred = preview_jobs[preview_jobs.stage.eq("E4")]
preview = preferred.iloc[0] if len(preferred) else preview_jobs.iloc[0] if len(preview_jobs) else None
SELECTED_RUN = preview.run if preview is not None else None  # replace with an explicit run path
SELECTED_JOB = preview.job_id if preview is not None else None  # paste a job_id from a table
SHOW_REFERENCE = False
CANDIDATE_INDEX = 0
if SELECTED_RUN and SELECTED_JOB:
    selected = catalog.row(SELECTED_RUN, SELECTED_JOB)
    print(selected[["run", "stage", "case_id", "arm", "status", "artifact_dir"]].to_string())
    display(HTML(image_gallery(pd.DataFrame([selected]), MAX_IMAGES)))
    try:
        panels = point_panels(catalog, selected, show_reference=SHOW_REFERENCE, candidate_index=CANDIDATE_INDEX)
        fig = point_figure(panels, MAX_POINTS_PER_CLOUD, INTERACTIVE_3D)
        if INTERACTIVE_3D:
            display(HTML(fig.to_html(full_html=False, include_plotlyjs=True)))
        else:
            display(fig)
            plt.close(fig)
    except (OSError, KeyError, ValueError) as exc:
        print("Preview unavailable:", exc)
else:
    print("No completed geometry artifacts in the current selection.")

## Interpreting the evidence

| Pattern | Architecture implication |
|---|---|
| B4 helps, B6 fails | Investigate the image-to-3D bridge and alignment |
| B6 helps, B2 fails | Investigate completion conditioning and preservation |
| Reranking helps, refinement hurts | Retain ranking; revisit refinement strength and contact constraints |
| Wrong template helps equally | Object-specific prior value remains unproven |
| K grows but selection does not improve | Inspect sample diversity and selection features |
| Gating removes damage by rejecting everything | Coverage and retained benefit do not support useful gating |
| Student matches teacher but is faster | Useful distillation; accuracy improvement remains unproven |
| Failures concentrate in one corruption or piece count | Narrow supported scope and investigate that failure mode |

These interpretations require paired source evidence and valid controls. Do not tune on locked-test results. See `PIPELINE_RESULTS_NOTEBOOK.md` for paths, artifact semantics, and limitations.